# H3L2 — Recurrence, convolution and attention

**Notebook 4 of this week.** Before 2017, sequence models were recurrent or
convolutional. We compare the three architectures, first by how far
information has to travel, then part by part. The parts follow the slides:

1. how information travels: the path length
2. recurrence: the gradient over many steps
3. convolution: reach limited by depth
4. sequential and parallel computation
5. the copy task: accuracy against distance
6. why the convolutional network fails even within its reach

**Colab:** Runtime → Run all. About three minutes on the CPU runtime; faster
on a GPU runtime, where Part 4 also shows the effect of parallel hardware.


In [ ]:
import math, os, time
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 13})          # readable on slides and in Colab
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)
C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"
FIGDIR = os.environ.get("SEMINAR_FIGDIR")


def save(fig, name):
    """Only when the slides are built: store the figure for the deck."""
    if FIGDIR:
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"), bbox_inches="tight", dpi=200)


## Part 1 — How information travels: the path length

For a token to influence another token $d$ positions later, its information
has to pass through a number of operations fixed by the architecture:

- a **recurrent** network reads one token per step and keeps a summary of
  everything so far, so the information passes through $d$ steps;
- a **convolutional** layer combines a window of $k$ neighbouring positions,
  so each layer reaches $k - 1$ positions further, and $d$ positions need
  about $d / (k - 1)$ layers;
- **attention** compares every position with every earlier position
  directly, in one step.

The table counts these operations.


In [ ]:
k = 3                                   # convolution kernel size
print("    d   rnn   cnn  attn")
for d in [2, 8, 32, 128, 1024]:
    cnn = math.ceil(d / (k - 1))
    print(f"{d:5d} {d:5d} {cnn:5d} {1:5d}")

ds = np.array([2, 8, 32, 128, 1024])
fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.loglog(ds, ds, "s-", color=C_TEAL, label="recurrent: steps")
ax.loglog(ds, np.ceil(ds / (k - 1)), "D-", color=C_PURPLE, label="convolution (k=3): layers")
ax.loglog(ds, np.ones_like(ds), "^-", color=C_BLUE, label="attention")
ax.set_xlabel("distance d"); ax.set_ylabel("operations on the path")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l2-p1"); plt.show()


**Exercise.** How many layers does a convolutional network with $k = 5$ need to reach $d = 128$?

<details><summary>Answer</summary>

$\lceil 128 / 4 \rceil = 32$ layers: with $k = 5$ each layer reaches 4 positions further.

</details>


## Part 2 — Recurrence: the gradient over many steps

A recurrent network updates a state $h_t = W_h h_{t-1} + W_x x_t$ one token
at a time. During training, the gradient that reaches a token $d$ steps back
has been multiplied by $W_h$ once per step, so its size behaves like
$\sigma^d$, where $\sigma$ is the largest absolute eigenvalue of $W_h$. We
measure this for a linear recurrence of width 64 over 128 steps.


In [ ]:
D, STEPS_BACK = 64, 128


def gradient_profile(sigma):
    rng = np.random.default_rng(0)      # the same matrices for every sigma
    W = rng.normal(size=(D, D)) / np.sqrt(D)
    W *= sigma / np.abs(np.linalg.eigvals(W)).max()   # set the largest eigenvalue
    U = rng.normal(size=(D, D)) / np.sqrt(D)
    out = np.empty(STEPS_BACK)
    g = np.ones(D)
    for t in range(STEPS_BACK):
        out[t] = np.linalg.norm(U.T @ g)
        g = W.T @ g   # one step back
    return out


dist = np.arange(STEPS_BACK)
print("size at d = 128, relative to d = 0")
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for sigma, c in [(1.1, C_RED), (1.0, C_GRAY), (0.9, C_BLUE)]:
    prof = gradient_profile(sigma)
    ax.semilogy(dist, prof, color=c, lw=2, label=f"σ = {sigma}")
    print(f"sigma = {sigma}: {prof[-1] / prof[0]:.1e}")
ax.semilogy(dist, 0.9 ** dist * gradient_profile(0.9)[0], color=C_BLUE, ls=":", lw=1,
            label="0.9^d (predicted)")
ax.set_xlabel("distance d back from the last step"); ax.set_ylabel("size of the gradient")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l2-p2"); plt.show()


**Exercise.** Add $\sigma = 0.95$ to the list. Predict the printed value from $\sigma^{128}$ before running.

<details><summary>Answer</summary>

$0.95^{128} = 1.4 \times 10^{-3}$; the notebook prints $1.1 \times 10^{-3}$. The rule $\sigma^d$ gives the rate at which the gradient shrinks once the largest eigenvalue dominates.

</details>


With $\sigma < 1$ the gradient from distant tokens vanishes, and with
$\sigma > 1$ it explodes. Gated recurrent networks (LSTM, GRU) extend the
usable range, but the effect of distance remains.


## Part 3 — Convolution: reach limited by depth

A stack of $L$ causal convolutional layers with kernel size $k$ sees exactly
$1 + L(k - 1)$ positions: the current one and $L(k-1)$ before it. We build
such a stack with random weights and compute the gradient of the last output
with respect to every input position. Outside the window the gradient is
exactly zero: no training can connect those positions to the output.


In [ ]:
def conv_stack(L, k=3, d=16):
    torch.manual_seed(0)
    convs = [nn.Conv1d(d, d, k) for _ in range(L)]
    def f(x):                                         # x: [1, d, T]
        for c in convs:
            x = torch.tanh(c(F.pad(x, (k - 1, 0))))   # left padding: causal
        return x
    return f

Tn = 64
reached = []
for L in (2, 4, 8):
    x = torch.randn(1, 16, 64,
                    requires_grad=True)
    y = conv_stack(L)(x)[0, :, -1]
    y.sum().backward()
    g = x.grad[0].norm(dim=0)
    reach = int((g > 0).sum())
    print(f"L = {L}: {reach} positions, {Tn - reach} zeros")
    reached.append((g > 0).numpy())

fig, ax = plt.subplots(figsize=(6.4, 2.4))
ax.imshow(np.array(reached), cmap="Purples", aspect="auto", vmin=0, vmax=1.4)
ax.set_yticks(range(3), ["L = 2", "L = 4", "L = 8"])
ax.set_xlabel("input position (the last output is at the right)")
ax.set_title("positions with a nonzero gradient", fontsize=10)
plt.tight_layout(); save(fig, "fig-t-w5-h3l2-p3"); plt.show()


**Exercise.** Change the kernel size to $k = 5$ (`conv_stack(L, k=5)`). Predict the number of positions for $L = 4$ before running.

<details><summary>Answer</summary>

$1 + L(k - 1) = 1 + 4 \cdot 4 = 17$ positions, as the notebook confirms.

</details>


## Part 4 — Sequential and parallel computation

Step $t$ of a recurrent network needs the state of step $t - 1$, so its steps
must run one after another. Attention computes all positions with a few
matrix products, which a GPU executes in parallel. We time one forward pass
of each at several sequence lengths.


In [ ]:
width = 128
scale = math.sqrt(width // 4)
rnn = nn.GRU(width, width, batch_first=True).to(DEV)
print("times in ms")
times = []
with torch.no_grad():
    for n in [128, 256, 512, 1024]:
        x = torch.randn(1, n, width, device=DEV)
        q = kk = v = torch.randn(1, 4, n, width // 4, device=DEV)
        for _ in range(3):                            # warm up, then keep the last timing
            t0 = time.perf_counter(); rnn(x)
            if DEV == "cuda": torch.cuda.synchronize()
            t_rnn = time.perf_counter() - t0
            t0 = time.perf_counter()
            s = q @ kk.transpose(-1, -2) / scale
            F.softmax(s, -1) @ v
            if DEV == "cuda": torch.cuda.synchronize()
            t_att = time.perf_counter() - t0
        print(f"N={n:4d}  rnn {t_rnn * 1e3:5.2f}  attn {t_att * 1e3:5.2f}")
        times.append((n, t_rnn * 1e3, t_att * 1e3))

Ns, tr, ta = zip(*times)
fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.loglog(Ns, tr, "s-", color=C_TEAL, label="recurrent (GRU)")
ax.loglog(Ns, ta, "^-", color=C_BLUE, label="attention")
ax.minorticks_off()
ax.set_xticks(Ns, [str(n) for n in Ns])
ax.set_xlabel("sequence length N"); ax.set_ylabel("time of one forward pass (ms)")
ax.set_title(f"device: {DEV}", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l2-p4"); plt.show()


The recurrent network performs fewer arithmetic operations, yet takes longer.
On a GPU the gap grows with the length, because the GPU computes the
$N \times N$ scores in parallel. On a CPU the gap shrinks as the length grows,
because the $N \times N$ work is no longer done in parallel. The exact ratios
depend on the hardware.


## Part 5 — The copy task: accuracy against distance

We train the three architectures on the copy task of H3L1, with about the
same number of parameters (about 114,000). The recurrent and convolutional
networks have width 92, the transformer width 64, because a transformer block
also contains an MLP.

| model | body |
|---|---|
| recurrent | 2-layer GRU |
| convolutional | 4 causal convolutions with kernel 3, a window of 9 positions |
| attention | the two-layer transformer of H3L1 |


In [ ]:
V, T, D_MODEL, D_BASE = 64, 96, 64, 92
BOS, SEP, PAD = 1, 2, 0
MAX_M, STEPS, BATCH = 40, 500, 64


def make_batch(n, m=None):
    ms = torch.full((n,), m, dtype=torch.long) if m else torch.randint(2, MAX_M + 1, (n,))
    x = torch.full((n, T), PAD, dtype=torch.long)
    mask = torch.zeros(n, T - 1, dtype=torch.bool)
    x[:, 0] = BOS
    for i in range(n):
        mi = int(ms[i])
        seg = torch.randint(4, V, (mi,))
        x[i, 1:1 + mi] = seg
        x[i, 1 + mi] = SEP
        x[i, 2 + mi:2 + 2 * mi] = seg
        mask[i, 1 + mi:1 + 2 * mi] = True
    return x.to(DEV), mask.to(DEV)


class RNNModel(nn.Module):
    def __init__(self, d=D_BASE):
        super().__init__()
        self.emb, self.rnn = nn.Embedding(V, d), nn.GRU(d, d, num_layers=2, batch_first=True)
        self.head = nn.Linear(d, V)
    def forward(self, x):
        return self.head(self.rnn(self.emb(x))[0])


class CNNModel(nn.Module):
    K = 3
    def __init__(self, d=D_BASE, n_layers=4):
        super().__init__()
        self.L = n_layers
        self.emb = nn.Embedding(V, d)
        self.convs = nn.ModuleList(nn.Conv1d(d, d, self.K) for _ in range(n_layers))
        self.head = nn.Linear(d, V)
    def forward(self, x):
        h = self.emb(x).transpose(1, 2)
        for c in self.convs:
            h = F.relu(c(F.pad(h, (self.K - 1, 0))))
        return self.head(h.transpose(1, 2))
    @property
    def window(self):
        return 1 + self.L * (self.K - 1)


class Block(nn.Module):
    def __init__(self, H=4):
        super().__init__()
        self.H, self.dh = H, D_MODEL // H
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(D_MODEL, D_MODEL, bias=False)
                                              for _ in range(4))
        self.mlp = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(),
                                 nn.Linear(4 * D_MODEL, D_MODEL))
    def forward(self, x):
        B, Tn, _ = x.shape
        h = self.ln1(x)
        split = lambda t: t.view(B, Tn, self.H, self.dh).transpose(1, 2)
        q, k, v = split(self.wq(h)), split(self.wk(h)), split(self.wv(h))
        s = q @ k.transpose(-1, -2) / math.sqrt(self.dh)
        s = s + torch.triu(torch.full((Tn, Tn), -torch.inf, device=x.device), 1)
        x = x + self.wo((F.softmax(s, -1) @ v).transpose(1, 2).reshape(B, Tn, D_MODEL))
        return x + self.mlp(self.ln2(x))


class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, D_MODEL), nn.Embedding(T, D_MODEL)
        self.blocks = nn.ModuleList([Block(), Block()])
        self.head = nn.Linear(D_MODEL, V)
    def forward(self, x):
        h = self.emb(x) + self.pos(torch.arange(x.shape[1], device=x.device))
        for b in self.blocks:
            h = b(h)
        return self.head(h)


def train(model, m=None, steps=STEPS, seed=1):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    for _ in range(steps):
        x, mask = make_batch(BATCH, m=m)
        target = x[:, 1:].clone(); target[~mask] = -100
        loss = F.cross_entropy(model(x)[:, :-1].reshape(-1, V), target.reshape(-1),
                               ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
    return model


def accuracy(model, m, n=512):
    with torch.no_grad():
        x, mask = make_batch(n, m=m)
        pred = model(x)[:, :-1].argmax(-1)
        return (((pred == x[:, 1:]) & mask).sum() / mask.sum()).item()


models = {"recurrent (GRU)": RNNModel(),
          "convolutional": CNNModel(),
          "attention": Transformer()}
for name, model in models.items():
    t0 = time.time()
    train(model.to(DEV))
    n_par = sum(p.numel() for p in model.parameters())
    print(f"{name:16s} {n_par:,} parameters ({time.time() - t0:.0f} s)")

dists = [2, 4, 6, 8, 12, 16, 24, 32, 40]
acc = {name: [accuracy(m, d) for d in dists] for name, m in models.items()}
short = {"recurrent (GRU)": "rnn", "convolutional": "cnn", "attention": "attn"}
print("m      2     8    24    40")
for name in models:
    row = [acc[name][dists.index(m)] for m in (2, 8, 24, 40)]
    print(f"{short[name]:4s}" + "".join(f"{a:6.2f}" for a in row))

fig, ax = plt.subplots(figsize=(7.2, 3.8))
for (name, a), c, mk in zip(acc.items(), [C_TEAL, C_PURPLE, C_BLUE], "sD^"):
    ax.plot(dists, a, mk + "-", color=c, label=name, ms=6)
ax.axvline(9, color=C_PURPLE, ls="--", lw=1)
ax.text(9.5, 0.55, "window of the\nconvolutional network", fontsize=8, color=C_PURPLE)
ax.axhline(1 / (V - 4), color=C_GRAY, ls=":", lw=1)
ax.set_xlabel("length m of the copied sequence"); ax.set_ylabel("accuracy on the second copy")
ax.set_ylim(-0.04, 1.06); ax.legend(frameon=False, loc="center right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h3l2-p5"); plt.show()


## Part 6 — Why the convolutional network fails even within its reach

The convolutional network stays near chance even for short sequences, inside
its window. Two experiments separate the possible reasons.

- **A. A fixed distance.** If every example has the same length, the token to
  copy is always the same number of positions back, and a fixed filter can
  shift the sequence by that amount.
- **B. A wider window.** With 20 layers the window covers 41 positions, enough
  for every length in the task.


In [ ]:
cnn = train(CNNModel().to(DEV), m=3)        # A
print(f"A. fixed m = 3: {accuracy(cnn, 3):.2f}")
cnn20 = CNNModel(n_layers=20).to(DEV)        # B
train(cnn20)
print(f"B. window {cnn20.window}, m = 16: "
      f"{accuracy(cnn20, 16):.2f}")


With a fixed distance the convolutional network copies perfectly, so its
capacity is sufficient. With a window wide enough for every distance it still
fails when the distance varies. A convolution applies the same filter at
every position and for every input, so it cannot choose where to look from
the content of the input. Attention computes its weights from the input,
separately for every example.


**Exercise.** Double the width of the recurrent network, `RNNModel(d=184)`. Does the accuracy at $m = 40$ improve, and by how much?

<details><summary>Answer</summary>

Only a little. With width 184 (432,096 parameters, about four times as many) the accuracy at $m = 40$ rises from 0.10 to 0.18. More width does not remove the decay with distance.

</details>


## What this notebook showed

| | path length | limit | on the copy task |
|---|---|---|---|
| recurrent | $d$ sequential steps | the gradient scales like $\sigma^d$ | accuracy falls with distance |
| convolutional | about $d/(k-1)$ layers | exactly zero outside the window; the same filter everywhere | near chance |
| attention | 1 | cost grows with $N^2$ | high at every distance |
